# 🔄 Replicación de Base de Datos Parroquia
## Generación de Scripts para Crear Base de Datos Idéntica

Esta sección te ayudará a crear scripts SQL completos para replicar exactamente esta base de datos en otro servidor.

# 📊 Análisis de Base de Datos Parroquia
## Usando Servidor MCP PostgreSQL

Este notebook permite analizar la base de datos del sistema parroquial usando el servidor MCP PostgreSQL para obtener insights y estadísticas importantes.

## 🔗 Configuración y Conexión

Verificamos la conexión a la base de datos y exploramos la estructura general.

In [ ]:
# Las consultas se realizarán usando el servidor MCP PostgreSQL
# No necesitamos importar librerías ya que usamos el servidor MCP directamente

print("📋 Notebook configurado para usar MCP PostgreSQL Server")
print("🔗 Base de datos: parroquia_db")
print("👤 Usuario: parroquia_user")
print("🏠 Host: localhost:5432")

## 📊 Estadísticas Generales del Sistema

Obtenemos un resumen general de todos los datos en el sistema.

In [ ]:
# Consulta para obtener estadísticas generales
query_estadisticas = """
SELECT 
    'Familias' as entidad, COUNT(*) as total FROM familias
UNION ALL
SELECT 
    'Personas' as entidad, COUNT(*) as total FROM personas
UNION ALL
SELECT 
    'Parroquias' as entidad, COUNT(*) as total FROM parroquias
UNION ALL
SELECT 
    'Municipios' as entidad, COUNT(*) as total FROM municipios
UNION ALL
SELECT 
    'Departamentos' as entidad, COUNT(*) as total FROM departamentos
UNION ALL
SELECT 
    'Encuestas' as entidad, COUNT(*) as total FROM encuestas
ORDER BY total DESC;
"""

print("🔍 Ejecutando consulta de estadísticas generales...")
print(f"Query: {query_estadisticas}")

## 👨‍👩‍👧‍👦 Análisis de Familias

Exploramos los datos de las familias registradas en el sistema.

In [ ]:
# Análisis detallado de familias
query_familias = """
SELECT 
    estado_encuesta,
    COUNT(*) as cantidad_familias,
    AVG(tamaño_familia) as promedio_tamaño,
    MIN(tamaño_familia) as tamaño_min,
    MAX(tamaño_familia) as tamaño_max
FROM familias 
GROUP BY estado_encuesta
ORDER BY cantidad_familias DESC;
"""

print("👨‍👩‍👧‍👦 Analizando datos de familias por estado de encuesta...")
print(f"Query: {query_familias}")

In [ ]:
# Distribución de familias por tamaño
query_tamaño_familias = """
SELECT 
    tamaño_familia,
    COUNT(*) as cantidad_familias,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM familias 
WHERE tamaño_familia IS NOT NULL
GROUP BY tamaño_familia
ORDER BY tamaño_familia;
"""

print("📊 Distribución de familias por tamaño...")
print(f"Query: {query_tamaño_familias}")

## 🏘️ Análisis Geográfico

Exploramos la distribución geográfica de las familias y parroquias.

In [ ]:
# Distribución geográfica de familias
query_geografia = """
SELECT 
    d.nombre_departamento,
    m.nombre_municipio,
    p.nombre as nombre_parroquia,
    COUNT(f.id_familia) as total_familias
FROM familias f
LEFT JOIN parroquias p ON f.id_parroquia = p.id_parroquia
LEFT JOIN municipios m ON f.id_municipio = m.id_municipio
LEFT JOIN departamentos d ON m.id_departamento = d.id_departamento
GROUP BY d.nombre_departamento, m.nombre_municipio, p.nombre
ORDER BY total_familias DESC
LIMIT 20;
"""

print("🗺️ Analizando distribución geográfica de familias...")
print(f"Query: {query_geografia}")

In [ ]:
# Familias sin asignación de parroquia
query_sin_parroquia = """
SELECT 
    COUNT(*) as familias_sin_parroquia,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM familias), 2) as porcentaje
FROM familias 
WHERE id_parroquia IS NULL;
"""

print("⚠️ Verificando familias sin asignación de parroquia...")
print(f"Query: {query_sin_parroquia}")

## 👥 Análisis de Personas

Exploramos los datos demográficos de las personas registradas.

In [ ]:
# Análisis demográfico por edad
query_demografico = """
SELECT 
    CASE 
        WHEN DATE_PART('year', AGE(fecha_nacimiento)) < 18 THEN 'Menores (0-17)'
        WHEN DATE_PART('year', AGE(fecha_nacimiento)) BETWEEN 18 AND 35 THEN 'Jóvenes (18-35)'
        WHEN DATE_PART('year', AGE(fecha_nacimiento)) BETWEEN 36 AND 60 THEN 'Adultos (36-60)'
        ELSE 'Mayores (+60)'
    END as grupo_edad,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM personas 
WHERE fecha_nacimiento IS NOT NULL
GROUP BY 
    CASE 
        WHEN DATE_PART('year', AGE(fecha_nacimiento)) < 18 THEN 'Menores (0-17)'
        WHEN DATE_PART('year', AGE(fecha_nacimiento)) BETWEEN 18 AND 35 THEN 'Jóvenes (18-35)'
        WHEN DATE_PART('year', AGE(fecha_nacimiento)) BETWEEN 36 AND 60 THEN 'Adultos (36-60)'
        ELSE 'Mayores (+60)'
    END
ORDER BY cantidad DESC;
"""

print("👥 Analizando distribución demográfica por grupos de edad...")
print(f"Query: {query_demografico}")

In [ ]:
# Análisis por sexo si está disponible
query_sexo = """
SELECT 
    s.nombre as sexo,
    COUNT(p.id_personas) as cantidad,
    ROUND(COUNT(p.id_personas) * 100.0 / SUM(COUNT(p.id_personas)) OVER(), 2) as porcentaje
FROM personas p
LEFT JOIN sexos s ON p.id_sexo = s.id_sexo
GROUP BY s.nombre
ORDER BY cantidad DESC;
"""

print("⚧ Analizando distribución por sexo...")
print(f"Query: {query_sexo}")

## 📋 Análisis de Encuestas

Revisamos el estado y progreso de las encuestas realizadas.

In [ ]:
# Estado de las encuestas
query_encuestas = """
SELECT 
    COUNT(*) as total_encuestas,
    COUNT(DISTINCT id_familia) as familias_encuestadas,
    MIN(fecha_creacion) as primera_encuesta,
    MAX(fecha_creacion) as ultima_encuesta
FROM encuestas;
"""

print("📋 Analizando estado de las encuestas...")
print(f"Query: {query_encuestas}")

## 🏠 Análisis de Vivienda y Servicios

Exploramos las condiciones de vivienda y servicios básicos.

In [ ]:
# Análisis de servicios básicos
query_servicios = """
SELECT 
    'Pozo Séptico' as servicio,
    COUNT(CASE WHEN pozo_septico = true THEN 1 END) as familias_con_servicio,
    COUNT(*) as total_familias,
    ROUND(COUNT(CASE WHEN pozo_septico = true THEN 1 END) * 100.0 / COUNT(*), 2) as porcentaje
FROM familias
UNION ALL
SELECT 
    'Letrina' as servicio,
    COUNT(CASE WHEN letrina = true THEN 1 END) as familias_con_servicio,
    COUNT(*) as total_familias,
    ROUND(COUNT(CASE WHEN letrina = true THEN 1 END) * 100.0 / COUNT(*), 2) as porcentaje
FROM familias
UNION ALL
SELECT 
    'Recolección de Basura' as servicio,
    COUNT(CASE WHEN disposicion_recolector = true THEN 1 END) as familias_con_servicio,
    COUNT(*) as total_familias,
    ROUND(COUNT(CASE WHEN disposicion_recolector = true THEN 1 END) * 100.0 / COUNT(*), 2) as porcentaje
FROM familias
ORDER BY porcentaje DESC;
"""

print("🏠 Analizando servicios básicos de las familias...")
print(f"Query: {query_servicios}")

## 🔍 Consultas Personalizadas

Espacio para realizar consultas específicas según necesidades particulares.

In [ ]:
# Plantilla para consultas personalizadas
query_personalizada = """
-- Escriba aquí su consulta personalizada
SELECT 
    'Ejemplo' as campo,
    COUNT(*) as total
FROM familias
LIMIT 5;
"""

print("🔧 Plantilla para consultas personalizadas lista")
print("Modifique la variable 'query_personalizada' con su consulta SQL")

## 📈 Resumen y Conclusiones

En esta sección resumiremos los hallazgos principales del análisis.

In [ ]:
# Resumen final
print("📊 RESUMEN DEL ANÁLISIS")
print("=" * 50)
print("")
print("🔍 Análisis completado sobre la base de datos parroquial")
print("📋 Tablas principales analizadas:")
print("   • Familias: Datos familiares y encuestas")
print("   • Personas: Información demográfica")
print("   • Parroquias: Distribución geográfica")
print("   • Servicios: Condiciones de vivienda")
print("")
print("✅ Para ejecutar las consultas, use el servidor MCP PostgreSQL")
print("🔗 Conexión: localhost:5432/parroquia_db")